# IT135IU · Course Project · D3: Analysis

**Topic 15 (own topic): Prioritizing Cardiovascular Screening Outreach, CDC BRFSS 2023**

| Member | Student ID | Role |
|---|---|---|
| Vũ Tiến Dũng | ITDSIU25011 | Product Owner |
| Mai Hoàng Gia Bảo | ITDSIU25005 | Data Engineer |
| Lê Nguyễn Trúc Phương | ITDSIU25029 | Data Analyst |
| Nguyễn Trần Phước Lộc | ITDSIU25020 | Visualisation Lead |
| Huỳnh Bá Dũng | ITDSIU25012 | Reproducibility Lead |
| Dương Minh Chính | ITDSIU25007 | Ethics and Privacy Officer |

**Decision question.** Which adult population groups should a public-health program prioritize for cardiovascular screening
and prevention outreach, based on combinations of major cardiovascular risk factors and reported cardiovascular disease burden?

**This notebook delivers (D3, analysis):** (1) weighted prevalence by number of risk factors with design-based confidence intervals,
(2) a statistical relationship between `RiskCount` and reported CVD, (3) an age / confounding check, (4) the Candidate Table of age x risk groups,
(5) the frozen table for the optional ML extension. Charts belong to the visualisation lead and the recommendation to the product owner.

*Data source: Centers for Disease Control and Prevention (CDC), Behavioral Risk Factor Surveillance System (BRFSS) 2023.
This analysis was produced by students of International University, VNU-HCM and has not been reviewed or endorsed by CDC, HHS or the U.S. Government.*

## 0 · Setup
Run the notebook from top to bottom (**Restart kernel → Run All**). Input: the clean dataset written by `notebooks/02_d2_data_profiling_cleaning.ipynb`
(`data/processed/brfss2023_clean.parquet`). The priority-rule thresholds below are the analyst's proposal; the product owner owns the rule.

In [1]:
from pathlib import Path
import hashlib, math
import numpy as np
import pandas as pd
from IPython.display import display

SEED = 42                       # no randomness is used in D3; kept for the reproducibility audit
np.random.seed(SEED)
pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 80)

ROOT    = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CLEAN   = ROOT / "data/processed/brfss2023_clean.parquet"
ML_PATH = ROOT / "data/processed/brfss2023_ml_frozen.parquet"
DOCS    = ROOT / "docs"
D2      = DOCS / "d2_outputs"
OUT     = DOCS / "d3_outputs"
OUT.mkdir(parents=True, exist_ok=True)

# Priority rule (section 6)
MIN_CASES        = 100    # at least this many respondents with reported CVD
MAX_REL_CI       = 0.25   # 95% CI half-width at most 25% of the group's own prevalence
MIN_BURDEN_SHARE = 5.0    # at least this % of the total weighted CVD burden
Z95 = 1.96

## 1 · Load the clean dataset and rebuild the analysis sample
The sample rule is the one from D2 (section 5.1): `CVD_ANY` known, all six risk flags known, age known. Unknown is never coded as "No".

In [2]:
assert CLEAN.exists(), f"Missing {CLEAN}. Run notebooks/02_d2_data_profiling_cleaning.ipynb (it writes this file) or copy it from the team drive."
df = pd.read_parquet(CLEAN)
sample = df.loc[df["IN_SAMPLE"]].copy().reset_index(drop=True)

sample["y"] = sample["CVD_ANY"].astype(float)
sample["w"] = sample["_LLCPWT"].astype(float)
sample["RiskCount"] = sample["RiskCount"].astype(int)
sample["RISK_BAND"] = pd.Categorical(pd.cut(sample["RiskCount"], bins=[-1, 1, 3, 6], labels=["0-1", "2-3", "4-6"]),
                                     categories=["0-1", "2-3", "4-6"], ordered=True)
sample["AGE_BAND"] = pd.Categorical(sample["AGE_BAND"], categories=["18-34", "35-44", "45-54", "55-64", "65+"], ordered=True)

flow_file = D2 / "d2_sample_flow.csv"
if flow_file.exists():
    expected = int(pd.read_csv(flow_file)["respondents"].iloc[-1])
    assert len(sample) == expected, f"sample has {len(sample):,} rows but D2 says {expected:,}"
print(f"Analysis sample: {len(sample):,} respondents (same as the last row of the D2 sample flow)")
print("Reported CVD, unweighted:", f"{sample['y'].mean()*100:.2f}%", "| weighted:", f"{(sample['w']*sample['y']).sum()/sample['w'].sum()*100:.2f}%")

Analysis sample: 321,978 respondents (same as the last row of the D2 sample flow)
Reported CVD, unweighted: 12.35% | weighted: 9.90%


## 2 · Survey-design helpers
BRFSS is a complex survey, so the standard error of a simple random sample would be too small. The helpers use **Taylor linearization** by stratum (`_STSTR`) and
primary sampling unit (`_PSU`). A group is estimated as a domain: respondents outside the group count as zero but their PSUs stay in the design.
Weighted prevalence = Σ(weight × CVD_ANY) / Σ(weight). The 95% CI uses 1.96 and, for prevalences, the logit scale so that it stays between 0 and 100%.

In [3]:
class Design:
    """Strata / PSU structure of the analysis sample (computed once)."""
    def __init__(self, strata, psu):
        strata = np.asarray(strata, dtype="int64"); psu = np.asarray(psu, dtype="int64")
        key = strata * 10_000_000_000 + psu
        self.psu_code, uniq = pd.factorize(key)
        self.n_psu = len(uniq)
        self.str_of_psu, str_uniq = pd.factorize(uniq // 10_000_000_000)
        self.n_str = len(str_uniq)
        self.n_h = np.bincount(self.str_of_psu, minlength=self.n_str)       # PSUs per stratum

    def cov(self, Z):
        """Design-based covariance of the totals of the row-level values Z (n x k)."""
        Z = np.asarray(Z, dtype=float)
        if Z.ndim == 1:
            Z = Z[:, None]
        k = Z.shape[1]
        T = np.column_stack([np.bincount(self.psu_code, weights=Z[:, j], minlength=self.n_psu) for j in range(k)])
        S = np.column_stack([np.bincount(self.str_of_psu, weights=T[:, j], minlength=self.n_str) for j in range(k)])
        D = T - (S / self.n_h[:, None])[self.str_of_psu]
        f = np.where(self.n_h > 1, self.n_h / np.maximum(self.n_h - 1, 1), 0.0)[self.str_of_psu]
        return (D * f[:, None]).T @ D

design = Design(sample["_STSTR"], sample["_PSU"])
Y, W = sample["y"].to_numpy(), sample["w"].to_numpy()
print(f"Design: {design.n_str:,} strata, {design.n_psu:,} PSUs, {int((design.n_h < 2).sum())} strata with a single PSU")

def prevalence(mask):
    """Weighted prevalence (proportion) of CVD_ANY in a domain, its linearized SE and a logit 95% CI."""
    mask = np.asarray(mask, dtype=bool)
    Wd = W[mask].sum(); p = (W[mask] * Y[mask]).sum() / Wd
    se = math.sqrt(design.cov(np.where(mask, W * (Y - p) / Wd, 0.0))[0, 0])
    if 0 < p < 1:
        s = se / (p * (1 - p)); logit = math.log(p / (1 - p))
        lo, hi = 1 / (1 + math.exp(-(logit - Z95 * s))), 1 / (1 + math.exp(-(logit + Z95 * s)))
    else:
        lo = hi = p
    return p, se, lo, hi

def group_table(by):
    """One row per group: n, cases, weighted prevalence with CI, weighted population and CVD burden (millions)."""
    by = [by] if isinstance(by, str) else list(by)
    rows = []
    for key, idx in sample.groupby(by, observed=True).indices.items():
        key = key if isinstance(key, tuple) else (key,)
        m = np.zeros(len(sample), dtype=bool); m[idx] = True
        p, se, lo, hi = prevalence(m)
        rows.append({**dict(zip(by, key)), "n_respondents": int(m.sum()), "n_cvd": int(Y[m].sum()),
                     "weighted_prev_pct": 100 * p, "ci_low_pct": 100 * lo, "ci_high_pct": 100 * hi,
                     "ci_halfwidth_pp": 100 * (hi - lo) / 2,
                     "pop_millions": W[m].sum() / 1e6, "cvd_burden_millions": (W[m] * Y[m]).sum() / 1e6})
    return pd.DataFrame(rows)

p_all, _, lo_all, hi_all = prevalence(np.ones(len(sample), dtype=bool))
print(f"Whole analysis sample: {100*p_all:.2f}% (95% CI {100*lo_all:.2f}-{100*hi_all:.2f})")

Design: 2,104 strata, 321,978 PSUs, 98 strata with a single PSU
Whole analysis sample: 9.90% (95% CI 9.68-10.12)


## 3 · Weighted prevalence by RiskCount, with confidence intervals
D2's table by `RiskCount` with a 95% CI added. The point estimates are checked against D2's table: they must be identical. `pop_millions` and `cvd_burden_millions`
describe the weighted **analysis sample**, not all U.S. adults.

In [4]:
by_risk = group_table("RiskCount")
d2_file = D2 / "d2_prevalence_by_riskcount.csv"
if d2_file.exists():
    ref = pd.read_csv(d2_file).set_index("RiskCount")
    diff = (by_risk.set_index("RiskCount")["weighted_prev_pct"] - ref["weighted_prev_pct"]).abs().max()
    assert diff < 1e-6, f"point estimates differ from D2 by {diff}"
    print(f"Check against the D2 table: largest difference {diff:.1e} percentage points")
FMT3 = {"n_respondents": "{:,}", "n_cvd": "{:,}", "weighted_prev_pct": "{:.1f}", "ci_low_pct": "{:.1f}", "ci_high_pct": "{:.1f}",
        "ci_halfwidth_pp": "{:.2f}", "pop_millions": "{:.1f}", "cvd_burden_millions": "{:.2f}"}
display(by_risk.style.format(FMT3).hide(axis="index"))
by_risk.to_csv(OUT / "d3_prevalence_by_riskcount_ci.csv", index=False)

Check against the D2 table: largest difference 7.1e-15 percentage points


RiskCount,n_respondents,n_cvd,weighted_prev_pct,ci_low_pct,ci_high_pct,ci_halfwidth_pp,pop_millions,cvd_burden_millions
0,"70,850","2,098",2.2,2.0,2.5,0.22,44.7,1.00
1,"88,218","6,311",5.4,5.1,5.7,0.31,51.0,2.75
2,"77,522","10,725",10.9,10.5,11.4,0.46,40.4,4.42
3,"51,063","10,501",18.6,17.9,19.4,0.77,25.2,4.71
4,"25,315","6,971",25.6,24.4,26.9,1.24,12.8,3.27
5,"8,193","2,841",33.3,31.1,35.6,2.24,4.2,1.39
6,817,318,39.7,33.2,46.5,6.62,0.4,0.16


**Reading the table.** Weighted reported CVD rises with every additional risk factor, from 2.2% with none to 39.7% with all six (same point estimates as D2).
The intervals show how precisely each level is estimated; levels with few respondents (RiskCount 6) have the widest ones.

## 4 · Statistical relationship: RiskCount and reported CVD
Survey-weighted logistic regression of `CVD_ANY` on `RiskCount` (0-6, treated as a count); exp(coefficient) is the odds ratio per additional risk factor.
Standard errors use the same strata / PSU linearization (sandwich form). **Model A** has `RiskCount` only (crude); **Model B** adds age band and sex, so the
change from A to B shows how much of the crude association is carried by age. The regression uses numpy only.

In [5]:
def svy_logit(X, names, iters=30):
    """Survey-weighted logistic regression with design-based standard errors. X includes the intercept column."""
    X = np.asarray(X, float); w = W / W.mean()               # rescaling the weights changes neither beta nor the sandwich SE
    beta = np.zeros(X.shape[1]); p0 = (w * Y).sum() / w.sum(); beta[0] = math.log(p0 / (1 - p0))
    for _ in range(iters):
        mu = 1 / (1 + np.exp(-(X @ beta)))
        step = np.linalg.solve(X.T @ (X * (w * mu * (1 - mu))[:, None]), X.T @ (w * (Y - mu)))
        beta += step
        if np.max(np.abs(step)) < 1e-9:
            break
    mu = 1 / (1 + np.exp(-(X @ beta)))
    A = X.T @ (X * (w * mu * (1 - mu))[:, None])
    B = design.cov(X * (w * (Y - mu))[:, None])
    V = np.linalg.inv(A) @ B @ np.linalg.inv(A)
    se = np.sqrt(np.diag(V))
    return pd.DataFrame({"term": names, "coef": beta, "se": se,
                         "odds_ratio": np.exp(beta), "or_ci_low": np.exp(beta - Z95 * se), "or_ci_high": np.exp(beta + Z95 * se),
                         "p_value": [math.erfc(abs(v) / math.sqrt(2)) for v in beta / se]})

ones = np.ones(len(sample))
risk = sample["RiskCount"].to_numpy(float)
age_d = pd.get_dummies(sample["AGE_BAND"], prefix="age", drop_first=True).astype(float)     # reference: 18-34
female = (sample["_SEX"].to_numpy() == 2).astype(float)                                      # CDC _SEX: 1 male, 2 female

mA = svy_logit(np.column_stack([ones, risk]), ["intercept", "RiskCount"])
mB = svy_logit(np.column_stack([ones, risk, age_d.to_numpy(), female]), ["intercept", "RiskCount", *age_d.columns, "female"])
mA.insert(0, "model", "A crude"); mB.insert(0, "model", "B adjusted (age band + sex)")
models = pd.concat([mA, mB], ignore_index=True)
display(models.style.format({"coef": "{:.4f}", "se": "{:.4f}", "odds_ratio": "{:.3f}", "or_ci_low": "{:.3f}", "or_ci_high": "{:.3f}", "p_value": "{:.4f}"}).hide(axis="index"))
models.to_csv(OUT / "d3_logit_riskcount.csv", index=False)

a = mA.query("term == 'RiskCount'").iloc[0]; b = mB.query("term == 'RiskCount'").iloc[0]
print(f"Crude OR per additional risk factor:    {a.odds_ratio:.2f} (95% CI {a.or_ci_low:.2f}-{a.or_ci_high:.2f})")
print(f"Adjusted OR per additional risk factor: {b.odds_ratio:.2f} (95% CI {b.or_ci_low:.2f}-{b.or_ci_high:.2f})")

model,term,coef,se,odds_ratio,or_ci_low,or_ci_high,p_value
A crude,intercept,-3.4366,0.0246,0.032,0.031,0.034,0.0000
A crude,RiskCount,0.6074,0.0088,1.836,1.804,1.868,0.0000
B adjusted (age band + sex),intercept,-4.5435,0.0838,0.011,0.009,0.013,0.0000
B adjusted (age band + sex),RiskCount,0.4746,0.0100,1.607,1.576,1.639,0.0000
B adjusted (age band + sex),age_35-44,0.6108,0.1029,1.842,1.506,2.253,0.0000
B adjusted (age band + sex),age_45-54,1.1727,0.0941,3.231,2.687,3.885,0.0000
B adjusted (age band + sex),age_55-64,1.7049,0.0902,5.501,4.610,6.565,0.0000
B adjusted (age band + sex),age_65+,2.3299,0.0868,10.277,8.670,12.182,0.0000
B adjusted (age band + sex),female,-0.4275,0.0272,0.652,0.618,0.688,0.0000


Crude OR per additional risk factor:    1.84 (95% CI 1.80-1.87)
Adjusted OR per additional risk factor: 1.61 (95% CI 1.58-1.64)


**Reading the table.** Each additional risk factor is associated with about 1.84 times the odds of reporting CVD (95% CI 1.80-1.87). After adjusting for age band and sex the
odds ratio falls to 1.61 (1.58-1.64): age explains part of the crude association but not all of it. Age band is the strongest term (65+ against 18-34: OR 10.3) and women have lower
odds than men (OR 0.65). A p-value printed as 0.0 is below 0.00005, so report it as p < 0.001; with this many respondents the odds ratio and its interval say more than the p-value.
This is an **association** in self-reported survey data, not evidence that the risk factors cause CVD.

## 5 · Age / confounding check
The same model fitted inside each age band (the full design is kept, so the standard errors stay correct), and the weighted prevalence by age band x risk band.

In [6]:
def svy_logit_domain(mask):
    """Crude RiskCount model fitted only on respondents with mask=True; PSUs outside the domain stay in the design."""
    mask = np.asarray(mask, bool)
    X = np.column_stack([ones, risk]); w = np.where(mask, W / W[mask].mean(), 0.0)
    beta = np.zeros(2); p0 = (w * Y).sum() / w.sum(); beta[0] = math.log(p0 / (1 - p0))
    for _ in range(30):
        mu = 1 / (1 + np.exp(-(X @ beta)))
        step = np.linalg.solve(X.T @ (X * (w * mu * (1 - mu))[:, None]), X.T @ (w * (Y - mu)))
        beta += step
        if np.max(np.abs(step)) < 1e-9:
            break
    mu = 1 / (1 + np.exp(-(X @ beta)))
    A = X.T @ (X * (w * mu * (1 - mu))[:, None]); B = design.cov(X * (w * (Y - mu))[:, None])
    V = np.linalg.inv(A) @ B @ np.linalg.inv(A)
    return beta, np.sqrt(np.diag(V))

rows = []
for band in sample["AGE_BAND"].cat.categories:
    m = (sample["AGE_BAND"] == band).to_numpy()
    beta, se = svy_logit_domain(m)
    rows.append({"age_band": band, "n_respondents": int(m.sum()), "n_cvd": int(Y[m].sum()),
                 "or_per_risk_factor": math.exp(beta[1]), "or_ci_low": math.exp(beta[1] - Z95 * se[1]), "or_ci_high": math.exp(beta[1] + Z95 * se[1])})
within_age = pd.DataFrame(rows)
display(within_age.style.format({"n_respondents": "{:,}", "n_cvd": "{:,}", "or_per_risk_factor": "{:.2f}", "or_ci_low": "{:.2f}", "or_ci_high": "{:.2f}"}).hide(axis="index"))
within_age.to_csv(OUT / "d3_logit_within_age.csv", index=False)

age_risk = group_table(["AGE_BAND", "RISK_BAND"])
pr = age_risk.pivot(index="AGE_BAND", columns="RISK_BAND", values="weighted_prev_pct")
pr["ratio 4-6 vs 0-1"] = pr["4-6"] / pr["0-1"]
print("Weighted reported CVD prevalence (%) by age band and risk band")
display(pr.style.format("{:.1f}"))

age_band,n_respondents,n_cvd,or_per_risk_factor,or_ci_low,or_ci_high
18-34,"39,455",542,1.91,1.66,2.20
35-44,"39,884","1,167",1.71,1.57,1.86
45-54,"46,203","2,873",1.91,1.81,2.01
55-64,"62,541","7,285",1.73,1.67,1.80
65+,"133,895","27,898",1.44,1.41,1.48


Weighted reported CVD prevalence (%) by age band and risk band


RISK_BAND,0-1,2-3,4-6,ratio 4-6 vs 0-1
AGE_BAND,,,,
18-34,0.9,2.8,10.9,11.8
35-44,2.0,4.7,12.9,6.5
45-54,2.4,8.9,22.5,9.6
55-64,4.7,14.5,28.6,6.1
65+,12.5,23.0,33.7,2.7


**Reading the tables.** The odds ratio per additional risk factor is above 1 in every age band (1.44 to 1.91), and every interval excludes 1, so the gradient is **not produced by age alone**.
It is lowest among those aged 65 and over (1.44), where reported CVD is already common. Absolute prevalence is much higher in the older bands, so age still decides *who* carries the burden (section 6).

## 6 · Candidate Table (age band x risk band)
One row per candidate group, the table the product owner chooses the 3-5 priority groups from. Besides prevalence with its CI, it gives the group's **burden** (weighted number of people with reported CVD)
and its share of the total, so that high prevalence alone does not decide. `stable` = at least `MIN_CASES` cases and a CI half-width of at most `MAX_REL_CI` x the group's prevalence.
`priority_candidate` = stable, the lower end of its CI above the overall prevalence, and at least `MIN_BURDEN_SHARE`% of the total burden.

In [7]:
cand = age_risk.copy()
cand["group_label"] = "Age " + cand["AGE_BAND"].astype(str) + " · " + cand["RISK_BAND"].astype(str) + " risk factors"
cand["share_of_total_burden_pct"] = 100 * cand["cvd_burden_millions"] / cand["cvd_burden_millions"].sum()
cand["share_of_population_pct"] = 100 * cand["pop_millions"] / cand["pop_millions"].sum()
cand["rank_by_prevalence"] = cand["weighted_prev_pct"].rank(ascending=False, method="min").astype(int)
cand["rank_by_burden"] = cand["cvd_burden_millions"].rank(ascending=False, method="min").astype(int)
cand["prevalence_vs_overall"] = cand["weighted_prev_pct"] / (100 * p_all)
cand["rel_ci_halfwidth"] = cand["ci_halfwidth_pp"] / cand["weighted_prev_pct"]
cand["stable"] = (cand["n_cvd"] >= MIN_CASES) & (cand["rel_ci_halfwidth"] <= MAX_REL_CI)
cand["priority_candidate"] = (cand["stable"] & (cand["ci_low_pct"] > 100 * p_all)
                              & (cand["share_of_total_burden_pct"] >= MIN_BURDEN_SHARE))
cand.insert(0, "group_id", [f"G{i:02d}" for i in range(1, len(cand) + 1)])
cols = ["group_id", "group_label", "AGE_BAND", "RISK_BAND", "n_respondents", "n_cvd", "weighted_prev_pct", "ci_low_pct", "ci_high_pct",
        "ci_halfwidth_pp", "pop_millions", "share_of_population_pct", "cvd_burden_millions", "share_of_total_burden_pct",
        "rank_by_prevalence", "rank_by_burden", "prevalence_vs_overall", "rel_ci_halfwidth", "stable", "priority_candidate"]
cand = cand[cols].sort_values("rank_by_burden").reset_index(drop=True)
FMT6 = {"n_respondents": "{:,}", "n_cvd": "{:,}", "weighted_prev_pct": "{:.1f}", "ci_low_pct": "{:.1f}", "ci_high_pct": "{:.1f}", "ci_halfwidth_pp": "{:.2f}",
        "pop_millions": "{:.2f}", "share_of_population_pct": "{:.1f}", "cvd_burden_millions": "{:.2f}", "share_of_total_burden_pct": "{:.1f}",
        "prevalence_vs_overall": "{:.2f}", "rel_ci_halfwidth": "{:.2f}"}
display(cand.style.format(FMT6).hide(axis="index"))
cand.to_csv(OUT / "d3_candidate_table.csv", index=False)
print(f"Overall weighted prevalence: {100*p_all:.2f}% | priority candidates:", ", ".join(cand.loc[cand["priority_candidate"], "group_id"]))

group_id,group_label,AGE_BAND,RISK_BAND,n_respondents,n_cvd,weighted_prev_pct,ci_low_pct,ci_high_pct,ci_halfwidth_pp,pop_millions,share_of_population_pct,cvd_burden_millions,share_of_total_burden_pct,rank_by_prevalence,rank_by_burden,prevalence_vs_overall,rel_ci_halfwidth,stable,priority_candidate
G14,Age 65+ · 2-3 risk factors,65+,2-3,"65,273","15,528",23.0,22.3,23.8,0.75,23.70,13.3,5.46,30.9,3,1,2.33,0.03,True,True
G15,Age 65+ · 4-6 risk factors,65+,4-6,"18,446","6,377",33.7,32.1,35.3,1.61,7.47,4.2,2.51,14.2,1,2,3.40,0.05,True,True
G13,Age 65+ · 0-1 risk factors,65+,0-1,"50,176","5,993",12.5,11.8,13.2,0.71,16.82,9.4,2.10,11.9,7,3,1.26,0.06,True,True
G11,Age 55-64 · 2-3 risk factors,55-64,2-3,"26,316","3,589",14.5,13.5,15.6,1.01,14.04,7.9,2.04,11.5,5,4,1.47,0.07,True,True
G12,Age 55-64 · 4-6 risk factors,55-64,4-6,"8,926","2,525",28.6,26.5,30.7,2.11,4.89,2.7,1.40,7.9,2,5,2.89,0.07,True,True
G08,Age 45-54 · 2-3 risk factors,45-54,2-3,"17,213","1,370",8.9,8.0,9.9,0.97,10.87,6.1,0.97,5.5,9,6,0.90,0.11,True,False
G09,Age 45-54 · 4-6 risk factors,45-54,4-6,"4,478",956,22.5,20.1,25.2,2.54,2.86,1.6,0.65,3.7,4,7,2.28,0.11,True,False
G10,Age 55-64 · 0-1 risk factors,55-64,0-1,"27,299","1,171",4.7,4.1,5.3,0.62,13.45,7.5,0.63,3.6,11,8,0.47,0.13,True,False
G05,Age 35-44 · 2-3 risk factors,35-44,2-3,"11,952",529,4.7,4.0,5.6,0.82,9.08,5.1,0.43,2.4,10,9,0.48,0.17,True,False
G04,Age 35-44 · 0-1 risk factors,35-44,0-1,"26,042",413,2.0,1.7,2.4,0.36,18.54,10.4,0.37,2.1,14,10,0.20,0.18,True,False


Overall weighted prevalence: 9.90% | priority candidates: G14, G15, G13, G11, G12


**Reading the table.** Five groups are priority candidates: G14 (65+, 2-3 risk factors), G15 (65+, 4-6), G13 (65+, 0-1), G11 (55-64, 2-3) and G12 (55-64, 4-6). Together they hold about 37.5% of the
weighted sample but about 76% of its reported-CVD burden. G09 (45-54, 4-6 risk factors) has a high prevalence (22.5%) but only 3.7% of the burden, and G08's interval (8.0-9.9%) lies at or below the overall level (about 9.9%),
so neither passes. G13 qualifies through age rather than through risk factors; whether to keep it is the product owner's decision. G02 and G03 are not stable (G03 has only 47 cases).
Burdens are millions of people **in the weighted sample**, not national totals.

### 6.1 · Do the priority groups depend on the thresholds?
The priority rule is re-applied with other values of `MIN_CASES` and `MAX_REL_CI` (the burden and prevalence conditions stay as above).

In [8]:
rows = []
for min_cases in (50, 100, 200):
    for max_rel in (0.20, 0.25, 0.30):
        s = (cand["n_cvd"] >= min_cases) & (cand["rel_ci_halfwidth"] <= max_rel)
        pc = s & (cand["ci_low_pct"] > 100 * p_all) & (cand["share_of_total_burden_pct"] >= MIN_BURDEN_SHARE)
        rows.append({"MIN_CASES": min_cases, "MAX_REL_CI": max_rel, "n_priority": int(pc.sum()), "groups": ", ".join(cand.loc[pc, "group_id"])})
sens = pd.DataFrame(rows)
display(sens.style.format({"MAX_REL_CI": "{:.2f}"}).hide(axis="index"))
sens.to_csv(OUT / "d3_sensitivity_priority_groups.csv", index=False)

MIN_CASES,MAX_REL_CI,n_priority,groups
50,0.20,5,"G14, G15, G13, G11, G12"
50,0.25,5,"G14, G15, G13, G11, G12"
50,0.30,5,"G14, G15, G13, G11, G12"
100,0.20,5,"G14, G15, G13, G11, G12"
100,0.25,5,"G14, G15, G13, G11, G12"
100,0.30,5,"G14, G15, G13, G11, G12"
200,0.20,5,"G14, G15, G13, G11, G12"
200,0.25,5,"G14, G15, G13, G11, G12"
200,0.30,5,"G14, G15, G13, G11, G12"


**Reading the table.** The same five groups are selected in all nine settings, so the list does not depend on these two thresholds. The burden threshold (5%) was not varied; it is what keeps G09 out.

## 7 · Freeze the table for the ML extension
The optional Logistic Regression / Random Forest starts only after this table is frozen. Target: `CVD_ANY`. The three components of the target and `_MICHD` must never be features (leakage).
The table is the same complete-case sample as above, so it has no missing values.

In [9]:
TARGET   = "CVD_ANY"
FEATURES = ["HTN_FLAG", "CHOL_FLAG", "DIABETES_FLAG", "OBESITY_FLAG", "SMOKING_FLAG", "INACTIVE_FLAG", "_AGE80", "_SEX"]
LEAKAGE  = ["MI_FLAG", "CHD_FLAG", "STROKE_FLAG", "_MICHD", "CVDINFR4", "CVDCRHD4", "CVDSTRK3"]
EXTRA    = ["RiskCount", "_LLCPWT", "_STSTR", "_PSU"]      # RiskCount = sum of the six flags (do not use with them); weight and design variables

assert not set(FEATURES) & set(LEAKAGE), "leakage variable among the features"
ml = sample[[TARGET] + FEATURES + EXTRA].copy()
assert ml[[TARGET] + FEATURES].notna().all().all(), "missing values in the frozen table"
ml[TARGET] = ml[TARGET].astype("int8")
ml.to_parquet(ML_PATH, index=False)
sha = hashlib.sha256(ML_PATH.read_bytes()).hexdigest()

contract = f"""# ML data contract

File: `data/processed/{ML_PATH.name}` (git-ignored, shared through the team drive) | Rows: {len(ml):,} | Columns: {ml.shape[1]} | SHA-256: `{sha}`

## Target
`{TARGET}`: 1 = ever told they had a heart attack, angina/coronary heart disease or stroke; 0 = none of the three (self-reported).
Positive share (unweighted): {ml[TARGET].mean()*100:.2f}%, so the classes are imbalanced: report precision, recall, F1 and ROC-AUC, not accuracy alone.

## Allowed features
{chr(10).join(f"- `{c}`" for c in FEATURES)}

## Never use as features (leakage)
{chr(10).join(f"- `{c}`" for c in LEAKAGE)}

## Other columns (not features)
`RiskCount` (sum of the six flags, redundant with them); `_LLCPWT`, `_STSTR`, `_PSU` (survey weight and design).

## Missing values
Complete-case sample from D2: no missing values, no imputation. Unknown answers were never coded as "No".
"""
(OUT / "ml_data_contract.md").write_text(contract, encoding="utf-8")
print(f"Frozen table: {len(ml):,} rows x {ml.shape[1]} columns -> {ML_PATH.name}")
print("SHA-256:", sha)

Frozen table: 321,978 rows x 13 columns -> brfss2023_ml_frozen.parquet
SHA-256: 90f7fac9e1a57e2ed0dd89878a476b51056af2dc79996408e6de2ddc751b881d


### What D3 does and does not establish
* **Shows:** reported CVD rises with the number of risk factors, in every age band, with design-based confidence intervals; and which age x risk groups combine a high prevalence with a large burden.
* **Does not show:** that risk factors cause CVD, individual clinical risk, or undiagnosed disease (the outcome is *already diagnosed* and self-reported). Results cover adults in the 48 states and DC of the 2023 public file.
* **Limitations:** complete-case selection (the analysis sample is older and has higher reported CVD than all respondents, D2 section 5.2); coarse risk-factor definitions (borderline blood pressure counts as "No", "never had cholesterol checked" is unknown);
  small groups have wide intervals; the priority thresholds are judgement calls (section 6.1 varies two of the three).